RAMAD public-package overview

This notebook reads the public QA corpus, previews the training-input structure, inspects the public spectrum cases, and lists the executable entry points. It uses only the Python standard library and does not call an external model service.


In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PACKAGE_ROOT = next((path for path in candidates if (path / "04_training_corpus_1000" / "RAMAD_domain_QA_1000.jsonl").is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or its 03_notebook directory.")

CORPUS_PATH = PACKAGE_ROOT / "04_training_corpus_1000" / "RAMAD_domain_QA_1000.jsonl"
SPECTRUM_PATH = PACKAGE_ROOT / "02_code_entrypoints" / "spectral" / "external_test_spectra_two_examples.csv"
ENTRY_POINTS = [
    PACKAGE_ROOT / "02_code_entrypoints" / "ramad_model" / "train_sft_lora.py",
    PACKAGE_ROOT / "02_code_entrypoints" / "ramad_model" / "run_inference.py",
    PACKAGE_ROOT / "02_code_entrypoints" / "ramad_rag" / "build_index.py",
    PACKAGE_ROOT / "02_code_entrypoints" / "ramad_rag" / "rag_qa.py",
    PACKAGE_ROOT / "benchmark" / "run_benchmark.py",
    PACKAGE_ROOT / "benchmark" / "aggregate_scores.py",
]
print(PACKAGE_ROOT)


In [ ]:
def load_jsonl(path):
    with path.open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]

qa_records = load_jsonl(CORPUS_PATH)
required_fields = {"id", "instruction", "input", "output", "topic", "language"}
assert len(qa_records) == 1000
assert all(required_fields == set(record) for record in qa_records)
print(len(qa_records))
print(dict(sorted(Counter(record["topic"] for record in qa_records).items())))
print(dict(sorted(Counter(record["language"] for record in qa_records).items())))


In [ ]:
record = qa_records[0]
parts = [f"Topic: {record['topic']}", f"Language: {record['language']}", "", f"Instruction:\n{record['instruction']}"]
if record["input"].strip():
    parts.extend(["", f"Input:\n{record['input']}"])
print("\n".join(parts))
print("\nTarget output:\n" + record["output"])
print(PACKAGE_ROOT / "02_code_entrypoints" / "ramad_model" / "train_sft_lora.py")


In [ ]:
with SPECTRUM_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    spectrum_rows = list(reader)
    spectrum_columns = [name for name in reader.fieldnames if name not in {"Category", "Conc"}]

print(len(spectrum_rows))
print(len(spectrum_columns))
print([(row["Category"], row["Conc"]) for row in spectrum_rows])


In [ ]:
for path in ENTRY_POINTS:
    print(path.relative_to(PACKAGE_ROOT), path.is_file())


In [ ]:
commands = [[sys.executable, str(path), "--help"] for path in ENTRY_POINTS]
for command in commands:
    print(" ".join(command))


Entry points

`02_code_entrypoints/ramad_model/` contains the training and inference scripts.

`02_code_entrypoints/ramad_rag/` contains the index-building and question-answering scripts.

`benchmark/` contains the common-question scoring workflow.

`run_public_package.ps1` checks the corpus, public spectrum cases, and command-line availability.
